In [75]:
import pandas as pd
from tqdm.std import tqdm
from openai import OpenAI
import re
import os
import json

In [76]:
input_data = "input_files/relationship_extraction_0001.csv"
output_data = "output_files/relationship_extraction_0001.xlsx"

In [ ]:
## Open AI API
os.environ["OPENAI_API_KEY"] = "your open ai api"

## common functions

In [78]:
def to_pascal_case(phrase):
    # Remove parentheses and the content inside them, e.g., (Unaudited)
    phrase = re.sub(r'\s*\([^)]*\)', '', phrase)
    
    # Split on non-alphanumeric characters
    words = re.split(r'[^a-zA-Z0-9]', phrase)
    
    # Capitalize each word and join
    return ''.join(word.capitalize() for word in words if word)

In [79]:
def read_xml(xml_path):
    with open(xml_path, "r", encoding="utf-8") as f:
        content = f.read()
        return content

In [ ]:
"""
    This function is for the relationship extraction task.
"""
def construct_annotation(df, results, task_name, base_data_path, rule_id="0001"):
    """
        the options of task_name are semantic_matching, relationship_extraction, and mathematical_reasoning
    """
    new_df = df.copy(deep=True)

    schema, presentation, calculation, definition, label, instance, us_gaap, true_answer, segmentation_path, parse_ok, input_col, outputByLLM = [[] for _ in range(12)]

    for res in results:
        folder_name = res.get("folder_name")
        prefix_path = os.path.join(base_data_path, folder_name)
        
        parse = False
        cal, pre, defn, lab, ins, sch, seg_path = [None] * 7
        
        ## special for relationship extraction task (DQC.US.0001)
        main_concept = res.get("main_concept")
        dimension_pair = res.get("dimension_pair")
        
        flattened_answers = []
        if main_concept:
            flattened_answers.append(main_concept)
        if dimension_pair and '=' in dimension_pair:
            flattened_answers.extend(dimension_pair.split('=', 1))
        
        answers = []
        if dimension_pair and '=' in dimension_pair:
            axis, member = dimension_pair.split('=', 1)
            answers.append((axis, "CombinationErr", member))

        if os.path.exists(prefix_path) and os.path.isdir(prefix_path):
            all_subfolders = [d for d in os.listdir(prefix_path) if os.path.isdir(os.path.join(prefix_path, d))]

            for subfolder_name in all_subfolders:
                full_path = os.path.join(prefix_path, subfolder_name)
                temp_cal, temp_pre, temp_defn, temp_lab, temp_ins, temp_sch = [None] * 6
                
                file_list = os.listdir(full_path)
                for file_name in file_list:
                    final_path = os.path.join(full_path, file_name)
                    if file_name.endswith("cal.xml"): temp_cal = read_xml(final_path)
                    elif file_name.endswith("pre.xml"): temp_pre = read_xml(final_path)
                    elif file_name.endswith("def.xml"): temp_defn = read_xml(final_path)
                    elif file_name.endswith("lab.xml"): temp_lab = read_xml(final_path)
                    elif file_name.endswith("htm.xml"): temp_ins = read_xml(final_path)
                    elif file_name.endswith("xsd"): sch = read_xml(final_path)
                
                is_appear = True
                ## check the definition linkbase
                definition_content = temp_defn or ""
                
                if not flattened_answers or not definition_content:
                    is_appear = False
                else:
                    for ans in flattened_answers:
                        if not (ans in definition_content or ans.replace(":", "_") in definition_content):
                            is_appear = False
                            break
               
                if is_appear:
                    parse = True
                    cal, pre, defn, lab, ins, sch = temp_cal, temp_pre, temp_defn, temp_lab, temp_ins, temp_sch
                    seg_path = os.path.join(folder_name, subfolder_name)
                    break 
        
        calculation.append(cal); presentation.append(pre); definition.append(defn); label.append(lab); instance.append(ins); schema.append(sch);
        true_answer.append(answers); us_gaap.append(None); parse_ok.append(parse); segmentation_path.append(seg_path); input_col.append(None); outputByLLM.append(res);
     
    new_df["schema"] = schema; new_df["presentation"] = presentation; new_df["calculation"] = calculation; new_df["definition"] = definition;
    new_df["label"] = label; new_df["instance"] = instance; new_df["us_gaap"] = us_gaap; new_df["input"] = input_col;
    new_df["true_answer"] = true_answer; new_df["segmentation_path"] = segmentation_path; new_df["parse_successful"] = parse_ok; new_df["LLM_Output"] = outputByLLM;

    return new_df

## load dqc messages for relationship extraction task (DQC.US.0001)

In [81]:
re_0001_df = pd.read_csv(input_data)

In [82]:
re_0001_df.head()

,assertion.code,assertion.detail,assertion.run-date,assertion.severity,assertion.source,assertion.type,entity.cik,entity.name,report.accepted-timestamp,report.accession,report.base-taxonomy,report.creation-software,report.document-type,report.entry-url,report.filing-date,report.filing-year,report.id,report.sec-url,report.sic-code,folder_name
0,DQC.US.0001.75,"2022-03-09 20:00:03,342 [DQC.US.0001.75] The c...",2022-03-10,ERROR,DQC,1,1383088,CYTTA CORP.,2022-01-12 13:46:00,0001477932-22-000209,US GAAP 2020,XBRL Document Created with XBRLMaster,10-K,http://www.sec.gov/Archives/edgar/data/1383088...,2022-01-12,2022,411725,https://www.sec.gov/Archives/edgar/data/138308...,7372,10k-cyca-20210930
1,DQC.US.0001.75,"2022-03-09 20:00:03,342 [DQC.US.0001.75] The c...",2022-03-10,ERROR,DQC,1,1383088,CYTTA CORP.,2022-01-12 13:46:00,0001477932-22-000209,US GAAP 2020,XBRL Document Created with XBRLMaster,10-K,http://www.sec.gov/Archives/edgar/data/1383088...,2022-01-12,2022,411725,https://www.sec.gov/Archives/edgar/data/138308...,7372,10k-cyca-20210930
2,DQC.US.0001.75,"2022-03-09 20:00:03,342 [DQC.US.0001.75] The c...",2022-03-10,ERROR,DQC,1,1383088,CYTTA CORP.,2022-01-12 13:46:00,0001477932-22-000209,US GAAP 2020,XBRL Document Created with XBRLMaster,10-K,http://www.sec.gov/Archives/edgar/data/1383088...,2022-01-12,2022,411725,https://www.sec.gov/Archives/edgar/data/138308...,7372,10k-cyca-20210930
3,DQC.US.0001.75,"2022-03-09 20:00:03,343 [DQC.US.0001.75] The c...",2022-03-10,ERROR,DQC,1,1383088,CYTTA CORP.,2022-01-12 13:46:00,0001477932-22-000209,US GAAP 2020,XBRL Document Created with XBRLMaster,10-K,http://www.sec.gov/Archives/edgar/data/1383088...,2022-01-12,2022,411725,https://www.sec.gov/Archives/edgar/data/138308...,7372,10k-cyca-20210930
4,DQC.US.0001.75,"2022-03-09 20:00:03,343 [DQC.US.0001.75] The c...",2022-03-10,ERROR,DQC,1,1383088,CYTTA CORP.,2022-01-12 13:46:00,0001477932-22-000209,US GAAP 2020,XBRL Document Created with XBRLMaster,10-K,http://www.sec.gov/Archives/edgar/data/1383088...,2022-01-12,2022,411725,https://www.sec.gov/Archives/edgar/data/138308...,7372,10k-cyca-20210930


## LLM judge assisstant

In [ ]:
Client = OpenAI()

In [84]:
re_0001_system_prompt = """You are an expert in XBRL taxonomies, tasked with parsing DQC validation messages for relationship errors (Rule DQC.US.0001), specifically an axis with an inappropriate member.

1. **Main Concept**: Extract the primary concept that is being dimensionally qualified. This is usually the first concept mentioned in the message.

2. **Dimension Pair**: From the 'Dimensions' field in the message, extract the full axis and member combination string **exactly as it appears**, including all prefixes.

3. **Reasoning**: Provide a short explanation that the member is unallowable for the specified axis, as described in the message.

Output your result as a single, structured JSON object in the following format:
```json
{
  "main_concept": "UnrealizedGainLossOnCashFlowHedgingInstruments",
  "dimension_pair": "us-gaap:FairValueByFairValueHierarchyLevelAxis=us-gaap:FairValueMeasurementsRecurringMember",
  "reason": "The message indicates the member is unallowable for the specified axis."
}
```"""

In [85]:
def get_response(dqc_message):
    completion = Client.chat.completions.create(
        model="gpt-4.1-mini",
        response_format={"type": "json_object"},
        messages=[
            {"role": "system", "content": re_0001_system_prompt},
            {"role": "user", "content": f"DQC validation message: {dqc_message} \\n Output:"}
        ]
    )
    return completion.choices[0].message.content

In [86]:
res = []
# to run on the full dataset, use the line below.
for _, row in tqdm(re_0001_df.iterrows(), total=re_0001_df.shape[0]):
# to test on only the first 10 examples, comment out the line above and uncomment the line below:
# for _, row in tqdm(re_0001_df.head(10).iterrows(), total=10):
    
    message = row["assertion.detail"]
    response = get_response(message)
    response = json.loads(response)

    # DQC 0001 messages do not have a statement name, so we only need the folder_name
    response["folder_name"] = row["folder_name"]
    res.append(response)

100%|██████████| 230/230 [05:05<00:00,  1.33s/it]


In [87]:
res[0]

{'main_concept': 'NetIncomeLoss',
 'dimension_pair': 'us-gaap:StatementEquityComponentsAxis=us-gaap:SeriesEPreferredStockMember',
 'reason': 'The message indicates the member SeriesEPreferredStockMember is unallowable for the StatementEquityComponentsAxis and should be used with the StatementClassOfStockAxis instead.',
 'folder_name': '10k-cyca-20210930'}

In [88]:
DATA_FOLDER_PATH = "./segmentation_data/relationship_extraction"

new_re_0001_df = construct_annotation(re_0001_df, res, 
                                     task_name="relationship_extraction", 
                                     base_data_path=DATA_FOLDER_PATH)

In [89]:
new_re_0001_df

,assertion.code,assertion.detail,assertion.run-date,assertion.severity,assertion.source,assertion.type,entity.cik,entity.name,report.accepted-timestamp,report.accession,...,calculation,definition,label,instance,us_gaap,input,true_answer,segmentation_path,parse_successful,LLM_Output
0,DQC.US.0001.75,"2022-03-09 20:00:03,342 [DQC.US.0001.75] The c...",2022-03-10,ERROR,DQC,1,1383088,CYTTA CORP.,2022-01-12 13:46:00,0001477932-22-000209,...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='UTF-8'?>\n<link:...,None,None,None,"[(us-gaap:StatementEquityComponentsAxis, Combi...",10k-cyca-20210930/StatementOfChangesInStockhol...,True,"{'main_concept': 'NetIncomeLoss', 'dimension_p..."
1,DQC.US.0001.75,"2022-03-09 20:00:03,342 [DQC.US.0001.75] The c...",2022-03-10,ERROR,DQC,1,1383088,CYTTA CORP.,2022-01-12 13:46:00,0001477932-22-000209,...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='UTF-8'?>\n<link:...,None,None,None,"[(us-gaap:StatementEquityComponentsAxis, Combi...",10k-cyca-20210930/StatementOfChangesInStockhol...,True,"{'main_concept': 'NetIncomeLoss', 'dimension_p..."
2,DQC.US.0001.75,"2022-03-09 20:00:03,342 [DQC.US.0001.75] The c...",2022-03-10,ERROR,DQC,1,1383088,CYTTA CORP.,2022-01-12 13:46:00,0001477932-22-000209,...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='UTF-8'?>\n<link:...,None,None,None,"[(us-gaap:StatementEquityComponentsAxis, Combi...",10k-cyca-20210930/StatementOfChangesInStockhol...,True,{'main_concept': 'StockIssuedDuringPeriodShare...
3,DQC.US.0001.75,"2022-03-09 20:00:03,343 [DQC.US.0001.75] The c...",2022-03-10,ERROR,DQC,1,1383088,CYTTA CORP.,2022-01-12 13:46:00,0001477932-22-000209,...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='UTF-8'?>\n<link:...,None,None,None,"[(us-gaap:StatementEquityComponentsAxis, Combi...",10k-cyca-20210930/StatementOfChangesInStockhol...,True,{'main_concept': 'CommonStockIssuedForCashShar...
4,DQC.US.0001.75,"2022-03-09 20:00:03,343 [DQC.US.0001.75] The c...",2022-03-10,ERROR,DQC,1,1383088,CYTTA CORP.,2022-01-12 13:46:00,0001477932-22-000209,...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='UTF-8'?>\n<link:...,None,None,None,"[(us-gaap:StatementEquityComponentsAxis, Combi...",10k-cyca-20210930/StatementOfChangesInStockhol...,True,{'main_concept': 'CommonStockIssuedForStockToB...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
225,DQC.US.0001.81,"2024-11-07 15:31:21,488 [DQC.US.0001.81] The ...",2024-11-07,ERROR,DQC,1,1320461,COOPER-STANDARD HOLDINGS INC.,2024-11-01 16:11:00,0001320461-24-000141,...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='UTF-8'?>\n<link:...,<?xml version='1.0' encoding='UTF-8'?>\n<xbrl ...,None,None,"[(StatementBusinessSegmentsAxis, CombinationEr...",10q-cps-20240930/SegmentReportingInformationon...,True,{'main_concept': 'StatementBusinessSegmentsAxi...
226,DQC.US.0001.81,"2024-11-07 15:31:21,488 [DQC.US.0001.81] The ...",2024-11-07,ERROR,DQC,1,1320461,COOPER-STANDARD HOLDINGS INC.,2024-11-01 16:11:00,0001320461-24-000141,...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='UTF-8'?>\n<link:...,<?xml version='1.0' encoding='UTF-8'?>\n<xbrl ...,None,None,"[(StatementBusinessSegmentsAxis, CombinationEr...",10q-cps-20240930/SegmentReportingInformationon...,True,"{'main_concept': 'SalesMember', 'dimension_pai..."
227,DQC.US.0001.74,"2024-08-28 21:07:06,428 [DQC.US.0001.74] The c...",2024-08-29,ERROR,DQC,1,797564,"HST Global, Inc.",2024-08-26 09:50:00,00

In [91]:
new_re_0001_df.to_excel(output_data, index=False)